# cuda-empty-cache — worked example 3: Verify empty_cache keeps a live tensor intact

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cuda-empty-cache`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`torch.cuda.empty_cache()` only returns cached blocks that have no live references; it can never free or corrupt a tensor you still hold. A good way to internalize this is to capture a tensor's value, call empty_cache, and confirm the tensor is byte-for-byte unchanged. This is true on CPU (where the call no-ops) and on GPU alike.

## Worked solution

**Step 1 — build a live tensor.** We seed and create `x`, then snapshot it with `x.clone()` so we have an independent reference value to compare against later.

**Step 2 — do real work that allocates.** We compute `y = x @ x.T`, which forces the allocator to hand out a fresh block for the result. `y` is live (we keep a reference), `x` is live.

**Step 3 — release the cache.** `t.cuda.empty_cache()` returns any unreferenced cached blocks to the driver. Crucially both `x` and `y` are still referenced, so neither is touched.

**Step 4 — prove nothing changed.** We assert `x` equals its snapshot exactly. Because empty_cache cannot disturb live memory, the equality holds. We return both the unchanged flag and the result `y` so the caller can confirm the work survived.

In [ ]:
def release_preserves(n: int):
    x = t.randn(n, n)
    snapshot = x.clone()
    y = x @ x.T
    t.cuda.empty_cache()
    unchanged = bool(t.equal(x, snapshot))
    return {'unchanged': unchanged, 'y': y}

t.manual_seed(0)
r = release_preserves(4)
print(r['unchanged'])
print(r['y'].shape)